In [562]:
import numpy as np
import pandas as pd

In [563]:
df=pd.read_csv("Churn_Modelling.csv")

In [564]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [565]:
df.shape

(10000, 14)

In [566]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


In [567]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [568]:
df.duplicated().sum()

np.int64(0)

In [569]:
(df['Exited']==0).value_counts()

Exited
True     7963
False    2037
Name: count, dtype: int64

In [570]:
df['Geography'].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [571]:
df['Gender'].value_counts()

Gender
Male      5457
Female    4543
Name: count, dtype: int64

In [572]:
df.drop(columns=["RowNumber","CustomerId","Surname"],inplace=True)

In [573]:
df=pd.get_dummies(df,columns=['Gender','Geography'],drop_first=True)

In [574]:
df.astype(int)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Gender_Male,Geography_Germany,Geography_Spain
0,619,42,2,0,1,1,1,101348,1,0,0,0
1,608,41,1,83807,1,0,1,112542,0,0,0,1
2,502,42,8,159660,3,1,0,113931,1,0,0,0
3,699,39,1,0,2,0,0,93826,0,0,0,0
4,850,43,2,125510,1,1,1,79084,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0,2,1,0,96270,0,1,0,0
9996,516,35,10,57369,1,1,1,101699,0,1,0,0
9997,709,36,7,0,1,0,1,42085,1,0,0,0
9998,772,42,3,75075,2,1,0,92888,1,1,1,0


In [575]:
df.columns

Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited', 'Gender_Male',
       'Geography_Germany', 'Geography_Spain'],
      dtype='str')

In [576]:
X=df.drop(columns=['Exited'])

In [577]:
y=df['Exited']

In [578]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [579]:
from sklearn.preprocessing import StandardScaler
scalar=StandardScaler()
X_train_scaled=scalar.fit_transform(X_train)
X_test_scaled=scalar.transform(X_test)

In [580]:
import torch.nn as nn
import torch.optim as optim


In [581]:
# model=nn.Sequential(
#     nn.Linear(11,3),
#     nn.Sigmoid(),
#     nn.Linear(3,1)
# )
model = nn.Sequential(
    nn.Linear(11, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)
# model = nn.Sequential(
#     nn.Linear(11, 64),
#     nn.LeakyReLU(),
#     nn.Linear(64, 32),
#     nn.LeakyReLU(),
#     nn.Linear(32, 1)
# )

In [582]:
loss_fn=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=0.03)

In [583]:
import torch
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)


In [584]:
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [585]:
X_train_tensor.shape, y_train_tensor.shape

(torch.Size([8000, 11]), torch.Size([8000, 1]))

In [586]:
for epoch in range(100):
    
    outputs=model(X_train_tensor)
    
    loss=loss_fn(outputs,y_train_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    
    optimizer.step()
    
    print(f"Epoch {epoch+1},Loss:{loss.item()}")

Epoch 1,Loss:0.6798871755599976
Epoch 2,Loss:0.6387637257575989
Epoch 3,Loss:0.5971608757972717
Epoch 4,Loss:0.5547712445259094
Epoch 5,Loss:0.5185106992721558
Epoch 6,Loss:0.49633708596229553
Epoch 7,Loss:0.488716185092926
Epoch 8,Loss:0.48623427748680115
Epoch 9,Loss:0.4800359606742859
Epoch 10,Loss:0.4700663685798645
Epoch 11,Loss:0.4595367908477783
Epoch 12,Loss:0.4509895145893097
Epoch 13,Loss:0.4456923305988312
Epoch 14,Loss:0.44322124123573303
Epoch 15,Loss:0.44248878955841064
Epoch 16,Loss:0.44236862659454346
Epoch 17,Loss:0.4420357346534729
Epoch 18,Loss:0.44107380509376526
Epoch 19,Loss:0.4395466446876526
Epoch 20,Loss:0.43775299191474915
Epoch 21,Loss:0.436073362827301
Epoch 22,Loss:0.4348294138908386
Epoch 23,Loss:0.43414220213890076
Epoch 24,Loss:0.43385395407676697
Epoch 25,Loss:0.43365105986595154
Epoch 26,Loss:0.4332403242588043
Epoch 27,Loss:0.43247920274734497
Epoch 28,Loss:0.43146347999572754
Epoch 29,Loss:0.4304169714450836
Epoch 30,Loss:0.4295901656150818
Epoch 31,

In [587]:
import torch
X_test = torch.tensor(X_test_scaled, dtype=torch.float32)

In [588]:
model.eval()

Sequential(
  (0): Linear(in_features=11, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
)

In [589]:
with torch.no_grad():
    outputs = model(X_test)




In [590]:
probs = torch.sigmoid(outputs)

In [591]:
preds = (probs > 0.5).float()

In [592]:
y_pred = preds.numpy()

In [593]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.86

In [594]:
print(y_train.value_counts())

Exited
0    6378
1    1622
Name: count, dtype: int64
